# LangChain RAG 實作

## 載入套件

In [1]:
import os
from openai import OpenAI
from langchain_openai import ChatOpenAI
from langchain_openai import OpenAIEmbeddings
import bs4
from langchain_unstructured import UnstructuredLoader
from langchain_core.documents import Document
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_openai import ChatOpenAI
from langgraph.graph import START, StateGraph
from typing_extensions import List, TypedDict
from dotenv import load_dotenv

## 載入組態檔，取得模型代碼

In [2]:
load_dotenv()
MODEL_ID = os.getenv('MODEL_ID')
MODEL_ID

'gpt-5.6-luna'

## 下載網路新聞

In [3]:
# 下載網路新聞
url = "https://www.ithome.com.tw/news/170362"
loader = UnstructuredLoader(web_url=url)
docs = loader.load()

In [4]:
# 分塊(Chunking)
text_splitter = RecursiveCharacterTextSplitter(chunk_size=300, chunk_overlap=50)
all_splits = text_splitter.split_documents(docs)
print(all_splits)

[Document(metadata={'category_depth': 0, 'link_texts': ['移至主內容'], 'link_urls': ['#main-content'], 'languages': ['kor'], 'filetype': 'text/html', 'url': 'https://www.ithome.com.tw/news/170362', 'category': 'Title', 'element_id': '2aac6d66caf7a6950c6fb62748a78a80'}, page_content='移至主內容'), Document(metadata={'category_depth': 0, 'link_texts': ['新聞'], 'link_urls': ['/tags/%E6%96%B0%E8%81%9E'], 'languages': ['kor'], 'filetype': 'text/html', 'url': 'https://www.ithome.com.tw/news/170362', 'category': 'Title', 'element_id': '912a0f7a508b6ebdb0cce5b3012dc09f'}, page_content='新聞'), Document(metadata={'category_depth': 0, 'languages': ['kor'], 'filetype': 'text/html', 'url': 'https://www.ithome.com.tw/news/170362', 'category': 'Title', 'element_id': '834b9ac535b2c834c23c256ee5c367b2'}, page_content='文化部提出文化藝術應用生成式AI指引，說明生成式AI應用潛在風險'), Document(metadata={'languages': ['kor'], 'filetype': 'text/html', 'parent_id': '834b9ac535b2c834c23c256ee5c367b2', 'url': 'https://www.ithome.com.tw/news/170362', 

In [5]:
# 此段內容與主題無關
all_splits[11]

Document(metadata={'languages': ['kor'], 'filetype': 'text/html', 'parent_id': '3ba12fb55b6c460039eab64fb590f1ef', 'url': 'https://www.ithome.com.tw/news/170362', 'category': 'UncategorizedText', 'element_id': 'ca576660aa637243c6c6c19ee26b21fa'}, page_content='生成式AI蒐集資料，需要複製資料，因此涉及「重製」，如果未經授權，且不符合著作權法第44到65條的合理使用規定，即可能侵害著作權，但目前著作權法中對「重製」他人作品，創作人輸入資料，以調校生成能力、增強結果，使用網路上的任何人可接觸的作品，作為GAI的資料，涉及著作的「重製」行為，但是目前沒有明文的合規使用規定，需視個案判定是否符合著作權法第65條第2項的一般合理使用規定，國內目前還沒有相關的司法判決案例。')

In [6]:
# 網路新聞內容
docs[0]

Document(metadata={'category_depth': 0, 'link_texts': ['移至主內容'], 'link_urls': ['#main-content'], 'languages': ['kor'], 'filetype': 'text/html', 'url': 'https://www.ithome.com.tw/news/170362', 'category': 'Title', 'element_id': '2aac6d66caf7a6950c6fb62748a78a80'}, page_content='移至主內容')

In [7]:
# 分成幾段(Chunks)、網路新聞字數
len(all_splits), len(docs[0].page_content)

(64, 5)

## 將網路新聞存入向量資料庫

In [8]:
from langchain_core.vectorstores import InMemoryVectorStore

# 使用 OpenAI 詞嵌入模型
embeddings = OpenAIEmbeddings()
# 使用存於記憶體的向量資料庫
vector_store = InMemoryVectorStore(embeddings)
# 將分塊內容存入向量資料庫
vector_store.add_documents(documents=all_splits[:11])

INFO: HTTP Request: POST https://api.openai.com/v1/embeddings "HTTP/1.1 200 OK"


['25f8e7e7-6016-40bb-9a2e-31cbc313d401',
 '3ea82dad-675f-4490-b0a8-0e5235990bf0',
 '7182de57-9e3b-48a4-9e8d-4b183385003f',
 '5f06c9b8-49b7-4cb6-a8d7-7f59520fa614',
 '86ddbb0f-b9e5-4c06-9077-788242ca7abb',
 'f2d57666-bc87-4e90-aa41-3ef7c35415ca',
 'c38f7df4-d96b-4fc2-8cd1-9d1caae30151',
 '2d4c461e-aa45-41f8-bb4c-ba4e6e48c203',
 '00adc2f6-d7f3-4af9-8dda-c3bef0399850',
 '3bb3cb2f-6d92-49f2-8db6-413e5a5fe655',
 '3bedf146-c80c-4560-bfa3-94de01149e3d']

## 查詢及生成

In [11]:
# 定義函數，提供資訊檢索
def retrieve(question):
    # 語意搜尋
    retrieved_docs = vector_store.similarity_search(question)
    return retrieved_docs

# 定義函數，使用提示範本合併上下文(Context)及提示
def augment_prompt(query, context):
    # 提示範本
    prompt_prefix = "Answer the question based only on the following context, " 
    prompt = prompt_prefix + f"Context: {context}\n\nQuestion: {query}\n\nAnswer:"
    return prompt

# 定義函數，進行 RAG 檢索、補強及生成
def generate(question):
    context = retrieve(question)
    docs_content = "\n\n".join(doc.page_content for doc in context)
    messages = augment_prompt(question, docs_content)
    print(messages)
    response = llm.invoke(messages)
    return response.content

In [12]:
# 建置OpenAI ChatBot
llm = ChatOpenAI(model=MODEL_ID)

# 測試
question = "著作權法是否保護畫風?"
response = generate(question)
print(response)

INFO: HTTP Request: POST https://api.openai.com/v1/embeddings "HTTP/1.1 200 OK"


Answer the question based only on the following context, Context: 藝術應用生成式AI指引涵蓋創作人、作品著作權人、文化藝術事業不同角度，從不同角度的立場，說明應用生成式AI所可能遭遇的法律風險。文化部指出，研擬指引的目的在於協助文化藝術工作者或事業，對應用生成式AI進行創作，可能涉及到的著作權、公平交易法等法律風險，指引為行政指導的性質，不具有法律拘束力，提醒讀者仍需注意GAI的法律規範，例如AI基本法，以及最新的司法判例。

文化部公布「藝術應用生成式AI指引」，針對生成式AI蒐集網路上的訓練資料，運用生成式AI產生特殊內容，例如吉卜力畫風內容是否違法等問題，提供指引參考資訊，說明潛在的法律風險。

文化部公布藝術應用生成AI指引，對外說明在文化藝術領域應用生成式AI，蒐集、產生內容、傳輸及出版所可能涉及的風險，包括侵害著作權，違反公平交易法、消保法等，供文化藝術產業運用生成式AI的指引參考。

文化部在指引中開宗明義提到，GAI的應用涉及對生成能力的調校及結果進行增強，從蒐集、掃描、歸類，到輸入提示，生成內容結果，將其散布至網路或是出版，涉及著作權法的「重製」、「改作」、「編輯」、「公開傳輸」。

Question: 著作權法是否保護畫風?

Answer:


INFO: HTTP Request: POST https://api.openai.com/v1/chat/completions "HTTP/1.1 200 OK"


著作權法原則上不保護抽象的「畫風」。因此，單純模仿或呈現特定藝術家的畫風，通常不當然構成著作權侵害。

但若生成內容進一步重製、改作或高度近似於特定受著作權保護的作品，仍可能涉及著作權侵害；此外，若造成市場混淆或攀附他人商譽，也可能涉及公平交易法等其他法律風險。
